# Notebook 2: Experiment 2 — Cross-Stock Prediction (80/20)
## A Comparative Analysis of BiLSTM and BiGRU for Stock Price Prediction

**Experiment:** Train on one stock's daily data, predict on another stock's daily test data.  
**Train/Test Split:** 80/20 (chronological)  
**Models:** BiLSTM, BiGRU, LSTM, GRU  
**Scaler:** ProportionScaler (÷ 10,501 BBCA ATH)  
**Metrics:** MSE, RMSE, MAE, MAPE, R² Score  


In [1]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '.')
from stock_prediction_utils import *

set_seed()
set_ieee_style()

DATA_DIR = 'dataset'

TRAIN_RATIO = 0.8
RATIO_LABEL = '80_20'
EXP_LABEL = f'Exp2_{RATIO_LABEL}'

os.makedirs(f'figures/{EXP_LABEL}', exist_ok=True)
os.makedirs(f'models/{EXP_LABEL}', exist_ok=True)
os.makedirs('results', exist_ok=True)

print(f"Experiment 2 - Cross-Stock Prediction (80/20)")


stock_prediction_utils.py loaded successfully!
  ProportionScaler max value: 10501.0
  Lookback: 60, Epochs: 200, Batch size: 64
  Architecture: 2 layers, 64 units, dropout=0.2
  Stocks: ['TLKM', 'BBCA', 'ASII', 'UNVR']
  Models: ['BiLSTM', 'BiGRU', 'LSTM', 'GRU']

GPU SETUP - CUDA Available
Number of GPUs detected: 1
  GPU 0: /physical_device:GPU:0

Memory growth enabled (dynamic allocation)
TensorFlow configured to use GPU


Device Configuration:
  GPUs available: 1
  CPUs available: 1
  TensorFlow will use GPU for computations
Experiment 2 - Cross-Stock Prediction (80/20)


In [2]:
# Load all daily data
print("Loading daily data...")
daily_data = load_all_daily_data(DATA_DIR)
print("\nAll daily data loaded!")


Loading daily data...
  TLKM: 5243 records, Date range: 2004-09-28 to 2025-12-31
  BBCA: 5244 records, Date range: 2004-09-28 to 2025-12-31
  ASII: 5244 records, Date range: 2004-09-28 to 2025-12-31
  UNVR: 5245 records, Date range: 2004-09-28 to 2025-12-31

All daily data loaded!


## Run All Cross-Stock Experiments

In [3]:
# ============================================================
# EXPERIMENT 2: Cross-stock prediction
# Train on Stock A, predict on Stock B's test data
# ============================================================
all_results = []
all_predictions = {}  # {(train_stock, test_stock): {model_type: (y_true, y_pred, dates)}}

for train_stock in STOCKS:
    for test_stock in STOCKS:
        if train_stock == test_stock:
            continue  # Skip same-stock (covered in Exp 1)
        
        pair_key = (train_stock, test_stock)
        print(f"\n{'#'*60}")
        print(f"# TRAIN: {train_stock} -> TEST: {test_stock}")
        print(f"{'#'*60}")
        
        # Prepare cross-stock data
        X_train, y_train, X_test, y_test, test_dates = prepare_cross_stock_data(
            daily_data[train_stock], daily_data[test_stock],
            train_ratio=TRAIN_RATIO, lookback=LOOKBACK
        )
        print(f"  X_train: {X_train.shape}, X_test: {X_test.shape}")
        
        all_predictions[pair_key] = {}
        
        for model_type in MODEL_TYPES:
            exp_name = f'{EXP_LABEL}_train_{train_stock}_test_{test_stock}'
            
            y_true_inv, y_pred_inv, metrics, history = train_and_evaluate(
                model_type=model_type,
                X_train=X_train, y_train=y_train,
                X_test=X_test, y_test=y_test,
                experiment_name=exp_name,
                save_dir=f'models/{EXP_LABEL}',
                epochs=EPOCHS, batch_size=BATCH_SIZE
            )
            
            result = {
                'Train_Stock': train_stock,
                'Test_Stock': test_stock,
                'Model': model_type,
                **metrics
            }
            all_results.append(result)
            all_predictions[pair_key][model_type] = (y_true_inv, y_pred_inv, test_dates)
            
            # Plot prediction
            plot_actual_vs_predicted(
                test_dates, y_true_inv, y_pred_inv,
                model_type, f'Train_{train_stock}_Test_{test_stock}',
                EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
            )

print("\n\nAll Experiment 2 (80/20) training complete!")



############################################################
# TRAIN: TLKM -> TEST: BBCA
############################################################
  X_train: (4134, 60, 1), X_test: (1049, 60, 1)

Training BiLSTM for: Exp2_80_20_train_TLKM_test_BBCA
  Train samples: 4134, Test samples: 1049
Epoch 1/200
58/59 [============================>.] - ETA: 0s - loss: 0.0017
Epoch 1: val_loss improved from inf to 0.00036, saving model to models/Exp2_80_20\Exp2_80_20_train_TLKM_test_BBCA_BiLSTM_best.keras
59/59 [==============================] - 12s 69ms/step - loss: 0.0017 - val_loss: 3.5599e-04
Epoch 2/200
59/59 [==============================] - ETA: 0s - loss: 1.5058e-04
Epoch 2: val_loss improved from 0.00036 to 0.00026, saving model to models/Exp2_80_20\Exp2_80_20_train_TLKM_test_BBCA_BiLSTM_best.keras
59/59 [==============================] - 2s 36ms/step - loss: 1.5058e-04 - val_loss: 2.6304e-04
Epoch 3/200
59/59 [==============================] - ETA: 0s - loss: 1.3823e-04
Epoch 3: val

KeyboardInterrupt: 

## Results Summary

In [ ]:
# ============================================================
# RESULTS TABLE
# ============================================================
results_df = pd.DataFrame(all_results)
print_results_table(results_df, f"Experiment 2 - Cross-Stock Prediction (80/20)")

results_df.to_csv(f'results/{EXP_LABEL}_results.csv', index=False)
print(f"Results saved to results/{EXP_LABEL}_results.csv")


## Visualizations

In [ ]:
# ============================================================
# HEATMAPS PER MODEL
# ============================================================
for metric in ['RMSE', 'MAE', 'MAPE (%)', 'R2']:
    plot_metrics_heatmap(
        results_df, metric, EXP_LABEL,
        row_col='Train_Stock', col_col='Test_Stock',
        save_dir=f'figures/{EXP_LABEL}'
    )

print("All heatmaps saved!")


In [ ]:
# ============================================================
# COMPARISON: All models for each train->test pair
# ============================================================
for train_stock in STOCKS:
    for test_stock in STOCKS:
        if train_stock == test_stock:
            continue
        pair_key = (train_stock, test_stock)
        if pair_key not in all_predictions:
            continue
        
        y_true = all_predictions[pair_key][MODEL_TYPES[0]][0]
        dates = all_predictions[pair_key][MODEL_TYPES[0]][2]
        preds = {mt: all_predictions[pair_key][mt][1] for mt in MODEL_TYPES if mt in all_predictions[pair_key]}
        
        plot_all_models_comparison(
            dates, y_true, preds,
            f'Train_{train_stock}_Test_{test_stock}',
            EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
        )

print("All comparison plots saved!")


In [ ]:
# ============================================================
# SUMMARY: BEST MODEL PER CROSS-STOCK PAIR
# ============================================================
print("\n" + "="*70)
print("  BEST MODEL PER PAIR (by RMSE)")
print("="*70)
for train_stock in STOCKS:
    for test_stock in STOCKS:
        if train_stock == test_stock:
            continue
        pair_data = results_df[
            (results_df['Train_Stock'] == train_stock) &
            (results_df['Test_Stock'] == test_stock)
        ]
        if pair_data.empty:
            continue
        best_idx = pair_data['RMSE'].idxmin()
        best = pair_data.loc[best_idx]
        print(f"  {train_stock} -> {test_stock}: {best['Model']} "
              f"(RMSE={best['RMSE']:.4f}, R²={best['R2']:.6f})")
